# Medium Analyzer: Ingestion Pipeline Notebook

This notebook mirrors the ingestion workflow from `ingestion.py` and explains each step.

Pipeline stages:
1. Load environment variables
2. Load source document
3. Split text into chunks
4. Build embeddings and ingest to Pinecone


In [ ]:
from pathlib import Path
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'ingestion.py').exists():
    PROJECT_DIR = Path('/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer')

load_dotenv(PROJECT_DIR / '.env')
print('SECTION 0: Environment loaded from .env')
print(f'Project directory: {PROJECT_DIR}')

## Section 1: Verify Required Configuration

Before ingestion, validate that all required environment variables are present.

In [ ]:
import os

required_keys = [
    'OPENAI_API_KEY',
    'LANGSMITH_API_KEY',
    'LANGSMITH_PROJECT',
    'LANGSMITH_TRACING',
    'INDEX_NAME',
    'PINECONE_API_KEY',
]

print('SECTION 1 RESULT:')
for key in required_keys:
    value = os.getenv(key)
    status = 'OK' if value else 'MISSING'
    print(f'- {key}: {status}')

## Section 2: Load Document

The `TextLoader` reads `mediumblog1.txt` and converts it into LangChain Document objects.

In [ ]:
from langchain_community.document_loaders import TextLoader

source_path = PROJECT_DIR / 'mediumblog1.txt'
loader = TextLoader(str(source_path))
documents = loader.load()

print('SECTION 2 RESULT:')
print(f'- Source path: {source_path}')
print(f'- Document count: {len(documents)}')
print(f"- First 180 chars: {documents[0].page_content[:180].replace(chr(10), ' ')}...")

## Section 3: Split Into Chunks

The splitter creates fixed-size chunks (`chunk_size=1000`, `chunk_overlap=0`) for embedding and indexing.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = splitter.split_documents(documents)

print('SECTION 3 RESULT:')
print(f'- Chunk count: {len(chunks)}')
if chunks:
    print(f'- First chunk length: {len(chunks[0].page_content)}')
    print(f"- First chunk preview: {chunks[0].page_content[:180].replace(chr(10), ' ')}...")

## Section 4: Pipeline Dry Run (Recommended First)

This calls `run_ingestion(..., dry_run=True)` from `ingestion.py` to verify the full workflow without external API calls.

In [ ]:
from ingestion import run_ingestion

summary = run_ingestion(source_path=source_path, dry_run=True)
print('SECTION 4 RESULT:')
print(summary)

## Section 5: Real Pinecone Ingestion (Explicit Pinecone Call)

This section performs the direct Pinecone write call from the notebook:
- initialize `OpenAIEmbeddings`
- call `PineconeVectorStore.from_documents(chunks, embeddings, index_name=...)`

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

import os

try:
    print('SECTION 5: Creating embeddings client...')
    embeddings = OpenAIEmbeddings(openai_api_key=os.environ.get('OPENAI_API_KEY'))
    print('SECTION 5: Writing chunks to Pinecone...')
    PineconeVectorStore.from_documents(
        chunks,
        embeddings,
        index_name=os.environ['INDEX_NAME'],
    )
    print(f"SECTION 5 RESULT: Ingested {len(chunks)} chunk(s) into index '{os.environ['INDEX_NAME']}'")
except Exception as e:
    print(f"SECTION 5 RESULT: FAILED -> {type(e).__name__}: {e}")

## Workflow Summary

- `ingestion.py` is the script entrypoint for CLI-style ingestion.
- This notebook is the guided, explainable workflow version.
- Both use the same logical steps: load -> split -> embed -> Pinecone upsert.
- Use dry-run first, then execute real ingestion when env and chunking look correct.